## **Clasificación de Comentarios Acusatorias**
### **Machine Learning - Proyecto Final**

**Integrantes:**
 - Isabella Tulcán
 - Daniel Andrade
 - Jarod Sierra

## **1. Introducción y Metodología**
### **1.1 Introducción**
El presente proyecto tiene como finalidad analizar los comentarios de los procesos de contratación pública en Ecuador utilizando modelos de aprendizaje supervisado tradicionales junto con técnicas de representación de texto (Regresión Logística y Random Forest) y feature extraction (TF-IDF y Transfer Learning), con el objetivo de detectar si los comentarios indican ser acusatorios o no.

Para cada variante de modelo se realizará su respectiva optimización de hiperpárametos mediante CV (cross-validation). Finalmente, se compará el rendimiento entre los distintos modelos con la finalidad de determinar aquel más idóneo.

### **1.2 Metodología**
- No se aplicarán técnicas de aumento de datos.
- El proceso de preprocesamiento del texto será el siguiente: corregir faltas ortográficas; remover stop words, urls y similares; finalmente, applicar lemmatization
- Entrenar modelos de Regresión Logística y Random Forest con técnicas de TF-IDF y Transfer Learning.
- Comparar el rendimiento de los modelos a través del test de hipótesis con $\alpha = 0.05$

## **2. Instalación de Dependencias y Configuración General**

In [ ]:
import os
import re
from pathlib import Path
from typing import Any
import warnings

import pandas as pd
import nltk
import stanza
import matplotlib.pyplot as plt
import numpy as np

from sklearn.manifold import TSNE
from symspellpy import SymSpell
from nltk.corpus import stopwords
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import RepeatedStratifiedKFold, GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    auc,
    classification_report,
    confusion_matrix,
)
warnings.filterwarnings('ignore')

nltk.download('stopwords', quiet=True)
RANDOM_STATE = 42

total_cores = os.cpu_count()
optimal_jobs = max(1, total_cores - 2)

## **3. Análisis exploratorio de los datos (EDA)**

In [ ]:
df = pd.read_excel('dataset.xlsx')
print(f"Forma (filas, columnas): {df.shape}")
print(f"\nNombres de columnas: {list(df.columns)}")
print(f"\nTipos de datos:\n{df.dtypes}")
print(f"\nPrimeras filas:")
df.head()

In [ ]:
#Estadísticas básicas
print(f"Valores nulos por columna:\n{df.isnull().sum()}")
print(f"\nRegistros duplicados (basado en preguntas): {df.duplicated(subset='pregunta').sum()}")
print("\nValores únicos por columna:")
print(df.nunique())

In [ ]:
counts = df['final_pregunta_isAcusatoria'].value_counts().reindex([0, 1], fill_value=0)

labels_map = {0: 'No Acusatoria', 1: 'Acusatoria'}
labels = [labels_map[i] for i in counts.index]

percentages = (counts / len(df)) * 100

print("\nDistribución de clases:")
print(pd.DataFrame({
    'Clase': labels,
    'Cantidad': counts.values,
    'Porcentaje (%)': percentages.round(2).values
}))

fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#2ecc71', '#e74c3c']
bars = ax.bar(labels, counts.values, color=colors, edgecolor='black')
max_count = counts.max()

for bar, count, pct in zip(bars, counts.values, percentages.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_count * 0.05,
        f'{count}\n({pct:.1f}%)',
        ha='center',
        va='bottom',
        fontweight='bold'
    )

ax.set_title('Distribución de clases')
ax.set_ylabel('Cantidad')
ax.set_xlabel('Clase')

plt.suptitle('Preguntas\nacusatorias vs no acusatorias', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

majority_class = labels[counts.values.argmax()]
minority_class = labels[counts.values.argmin()]
ratio = counts.max() / counts.min() if counts.min() > 0 else float('inf')

print(f"\nClase mayoritaria: {majority_class} ({counts.max()} registros, {percentages.max():.2f}%)")
print(f"Clase minoritaria: {minority_class} ({counts.min()} registros, {percentages.min():.2f}%)")
print(f"Ratio de desbalance: {ratio:.2f}:1")

# Interpretación
if percentages.min() < 20:
    print("Conclusión: existe un desbalance fuerte entre clases.")
elif percentages.min() < 40:
    print("Conclusión: existe un desbalance moderado entre clases.")
else:
    print("Conclusión: las clases están relativamente balanceadas.")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
vote_counts = df['sum_pregunta_isAcusatoria'].value_counts().sort_index()
percentages = (vote_counts / len(df)) * 100
colors = []
max_votes = vote_counts.index.max()

for v in vote_counts.index:
    if v == 0:
        colors.append('#2ecc71') 
    elif v == max_votes:
        colors.append('#e74c3c') 
    else:
        colors.append('#f1c40f')

bars = ax.bar(vote_counts.index, vote_counts.values, color=colors, edgecolor='black')
for bar, count, pct in zip(bars, vote_counts.values, percentages.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(vote_counts.values) * 0.02,
        f'{count}\n({pct:.1f}%)',
        ha='center',
        va='bottom',
        fontweight='bold'
    )

ax.set_title('Distribución de votos de anotadores')
ax.set_xlabel('Número de anotadores que marcaron como acusatoria')
ax.set_ylabel('Cantidad de preguntas')

plt.tight_layout()
plt.show()

In [ ]:
df['char_length'] = df['pregunta'].astype(str).str.len()
df['word_count'] = df['pregunta'].astype(str).str.split().str.len()

df['clase_nombre'] = df['final_pregunta_isAcusatoria'].map({
    0: 'No Acusatoria',
    1: 'Acusatoria'
})

summary = df.groupby('clase_nombre')[['char_length', 'word_count']].agg(['mean', 'median', 'std']).round(2)
print("Resumen de longitud por clase:")
print(summary)

fig, axes = plt.subplots(1, 2, figsize=(10,5))

for clase, color in zip(['No Acusatoria', 'Acusatoria'], ['#2ecc71', '#e74c3c']):
    subset = df[df['clase_nombre'] == clase]
    axes[0].hist(subset['word_count'], bins=30, alpha=0.6, label=clase, color=color)

axes[0].set_title('Distribución de longitud (palabras)')
axes[0].legend()

mean_words = df.groupby('clase_nombre')['word_count'].mean()

axes[1].bar(mean_words.index, mean_words.values, color=['#2ecc71', '#e74c3c'])

for i, v in enumerate(mean_words):
    axes[1].text(i, v + 0.2, f'{v:.1f}', ha='center', fontweight='bold')

axes[1].set_title('Promedio de palabras por clase')

plt.tight_layout()
plt.show()

In [ ]:
texto = df["pregunta"].fillna("").astype(str)

print("Inspección de normalización del texto")

vacios = (texto.str.strip() == "").sum()
print(f"Textos vacíos o con solo espacios: {vacios} / {len(df)}")

mayusculas = texto.str.contains(r"[A-Z]", regex=True).sum()
print(f"Textos con mayúsculas: {mayusculas} / {len(df)}")

tildes = texto.str.contains(r"[áéíóúÁÉÍÓÚ]", regex=True).sum()
print(f"Textos con tildes: {tildes} / {len(df)}")

enie = texto.str.contains(r"[ñÑ]", regex=True).sum()
print(f"Textos con ñ: {enie} / {len(df)}")

numeros = texto.str.contains(r"[0-9]", regex=True).sum()
print(f"Textos con números: {numeros} / {len(df)}")

urls = texto.str.contains(r"http[s]?://|www\.", regex=True).sum()
print(f"Textos con URLs: {urls} / {len(df)}")

signos = texto.str.contains(r"[¿?¡!]", regex=True).sum()
print(f"Textos con signos de interrogación/exclamación: {signos} / {len(df)}")


# Longitud (corregido)
longitud = texto.str.len()

muy_cortas = (longitud < 5).sum()
print(f"\nPreguntas muy cortas (<5 caracteres): {muy_cortas} / {len(df)}")

# Palabras acusatorias
patron_acusatorio = r"\b(?:robo|robó|corrupción|corrupcion|culpable|responsable|coima|soborno|fraude|delito|acusado|acusación|acusacion)\b"

acusatorias_keywords = texto.str.contains(
    patron_acusatorio, case=False, regex=True
).sum()

print(f"Textos con palabras potencialmente acusatorias: {acusatorias_keywords} / {len(df)}")

### **Conclusiones Del Análisis Exploratorio de los Datos (EDA)**

#### **Conclusiones Individuales**

1. El dataset se encuentra limpio a nivel estructural, ya que no presenta valores nulos, lo que permite avanzar directamente al análisis y modelado sin necesidad de aplicar técnicas de imputación.
2. Se observa una baja redundancia en las preguntas, lo cual es positivo para modelos de procesamiento de lenguaje natural, ya que favorece la diversidad de ejemplos y reduce el riesgo de sobreajuste.
3. La variable final_pregunta_isAcusatoria presenta un fuerte desbalance (33:1), lo que implica que métricas como accuracy pueden resultar engañosas. Por ello, será necesario utilizar métricas como precision, recall y F1-score, así como técnicas como class_weight='balanced'.
4. La mayoría de las preguntas no son consideradas acusatorias por los anotadores, lo que confirma la baja frecuencia de esta clase. Asimismo, la presencia de algunos desacuerdos sugiere posibles casos de ambigüedad en la clasificación.
5. Las preguntas acusatorias tienden a ser significativamente más largas, lo que sugiere que incluyen mayor nivel de detalle, contexto o argumentación.
6. El texto no se encuentra completamente normalizado, presentando variabilidad en el uso de mayúsculas, tildes y signos, por lo que será necesario aplicar preprocesamiento adecuado.
7. Las preguntas acusatorias no se identifican únicamente mediante palabras clave, sino que dependen del contexto y el tono, lo que sugiere la necesidad de modelos que capturen información semántica más compleja.

#### **Conclusión General**

El análisis exploratorio revela que el dataset presenta una alta diversidad textual y ausencia de valores nulos, lo que facilita su uso para modelado. Sin embargo, se identifica un fuerte desbalance en la variable objetivo, donde las preguntas acusatorias representan apenas el 2.94% del total. Este desbalance implica la necesidad de utilizar métricas robustas y técnicas específicas durante el entrenamiento del modelo.

Adicionalmente, se observa que las preguntas acusatorias tienden a ser significativamente más largas que las no acusatorias, lo que sugiere diferencias en la estructura y complejidad del lenguaje utilizado. Este hallazgo indica que la longitud del texto puede ser una variable relevante para la clasificación.

Finalmente, el análisis textual muestra que la identificación de preguntas acusatorias no depende únicamente de palabras clave, sino del contexto general de la pregunta, lo que sugiere la necesidad de aplicar técnicas de procesamiento de lenguaje natural más avanzadas.

## **4. Preprocesamiento del texto**

In [ ]:
df.drop(columns=['contract_id', 'pregunta_id', 'sum_pregunta_isAcusatoria', 'char_length', 'word_count', 'clase_nombre'], inplace=True)
df.drop_duplicates(subset=['pregunta', 'final_pregunta_isAcusatoria'], inplace=True)

print(f"Forma (filas, columnas): {df.shape}")

In [ ]:
def correct_text(text) -> str:
    sym_spell = SymSpell()
    sym_spell.load_dictionary(
        "es_merged_50k.txt", term_index=0, count_index=1, separator="\t"
    )

    return sym_spell.word_segmentation(text).corrected_string


def remove_stopwords(text) -> str:
    spanish_stopwords = set(stopwords.words("spanish")) - {"no", "si", "sí"}

    # Remover URLs
    text = re.sub(r"http\S+|www\.\S+", "", text)
    # Mantener letras en español, números y espacios
    text = re.sub(r"[^A-Za-z\sñÑáéíóúÁÉÍÓÚüÜ]", " ", text)
    # Normalizar espacios
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    tokens = [t for t in tokens if t not in spanish_stopwords and len(t) > 2]

    return " ".join(tokens)


def lemmatization(text: str) -> str:
    nlp = stanza.Pipeline(lang="es", processors="tokenize,lemma", verbose=False)
    doc = nlp(text)
    return " ".join([word.lemma for sent in doc.sentences for word in sent.words])

def count_words(column: str):
    words: set[str] = set()

    for sentence in df['column'].tolist():
        sentence: str = sentence

        for word in sentence.split():
            words.add(word)

    print(f"Number of unique words in '{column}': {len(words)}")

def preprocess_text() -> pd.DataFrame:
    if Path("dataset-clean.xlsx").is_file():
        print("Dataset is already cleaned!")
        return pd.read_excel('dataset-clean.xlsx',index_col=0)
    
    df['pregunta'] = df['pregunta'].str.lower()
    
    df["pregunta_corregida"] = df["pregunta"].apply(correct_text)
    count_words('pregunta_corregida')

    df['pregunta_limpia'] = df['pregunta_corregida'].apply(remove_stopwords)
    count_words('pregunta_limpia')

    df['pregunta_lemma'] = df['pregunta_limpia'].apply(lemmatization)
    count_words('pregunta_lemma')

    # Check for duplicateds after cleaning
    for question in (df[df.duplicated(subset='pregunta_lemma')])['pregunta_lemma'].tolist():
        filtered = df[df['pregunta_lemma'] == question] 

        display(filtered[['pregunta', 'final_pregunta_isAcusatoria']])

        for i in filtered.index:
            print(filtered.loc[i]['pregunta'])
        
        print() 
    
    df.drop_duplicates(subset=['pregunta_lemma'], inplace=True)
    df.save_to_excel('dataset-clean.xlsx')   
    print("Dataset cleaned!")
    
    return df

In [ ]:
df = preprocess_text()
print(df['final_pregunta_isAcusatoria'].value_counts())

Como podemos observar, tras realizar el preprocesamiento del texto, se encontraron más registros duplicados. Inicialmente, estos no pudieron ser detectados puesto que continen ligeras variaciones, como la presencia de un caracter adicional. Sin embargo, el contenido es idéntico.

## **5. Entrenamiento de Modelos**

In [ ]:
class SentenceTransformerEncoder(BaseEstimator, TransformerMixin):
    """Encoder SBERT compatible con sklearn, con cache global."""

    def __init__(
        self, model_name="paraphrase-multilingual-MiniLM-L12-v2", batch_size=32
    ):
        self.model_name = model_name
        self.batch_size = batch_size
        self._model = None
        self._cache: dict = {}

    def _ensure_model(self):
        if self._model is None:
            self._model = SentenceTransformer(self.model_name)

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        self._ensure_model()
        X_str = [str(x) for x in X]
        missing = list({x for x in X_str if x not in self._cache})
        if missing:
            embs = self._model.encode(
                missing,
                convert_to_numpy=True,
                show_progress_bar=False,
                batch_size=self.batch_size,
            )
            for t, e in zip(missing, embs):
                self._cache[t] = e
        return np.vstack([self._cache[x] for x in X_str])

In [ ]:
X_raw = df["pregunta_corregida"]
X_lemma = df["pregunta_lemma"]
y = df["final_pregunta_isAcusatoria"]

X_train_lemma, X_test_lemma, y_train, y_test = train_test_split(
    X_lemma, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

X_train_raw, X_test_raw, _, _ = train_test_split(
    X_raw, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)

### **Visualización TSNE**

In [ ]:
vectorizer = TfidfVectorizer(sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(X_train_lemma)
X_test_tfidf = vectorizer.transform(X_test_lemma)

In [ ]:
tfidf_2d = TSNE(
    n_components=2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
).fit_transform(X_train_tfidf)

fig, ax = plt.subplots(figsize=(7, 5))
mask_neg = y_train == 0
mask_pos = y_train == 1
ax.scatter(
    tfidf_2d[mask_neg, 0],
    tfidf_2d[mask_neg, 1],
    c="steelblue",
    alpha=0.3,
    s=10,
    label=f"No acusatoria (n={mask_neg.sum()})",
)
ax.scatter(
    tfidf_2d[mask_pos, 0],
    tfidf_2d[mask_pos, 1],
    c="red",
    alpha=0.9,
    s=40,
    edgecolors="darkred",
    linewidths=0.5,
    label=f"Acusatoria (n={mask_pos.sum()})",
)
ax.set_title("t-SNE de TF-IDF — Train set (label como color)")
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
transformer = SentenceTransformerEncoder()
transformer._ensure_model()

X_train_encoded = transformer.transform(X_train_raw)
X_test_encoded = transformer.transform(X_test_raw)

In [ ]:
emb_2d = TSNE(
    n_components=2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
).fit_transform(X_train_encoded)

fig, ax = plt.subplots(figsize=(7, 5))
mask_neg = y_train == 0
mask_pos = y_train == 1
ax.scatter(
    emb_2d[mask_neg, 0],
    emb_2d[mask_neg, 1],
    c="steelblue",
    alpha=0.3,
    s=10,
    label=f"No acusatoria (n={mask_neg.sum()})",
)
ax.scatter(
    emb_2d[mask_pos, 0],
    emb_2d[mask_pos, 1],
    c="red",
    alpha=0.9,
    s=40,
    edgecolors="darkred",
    linewidths=0.5,
    label=f"Acusatoria (n={mask_pos.sum()})",
)
ax.set_title("t-SNE de embeddings SBERT — Train set (label como color)")
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def evaluate_classifier(
    model, X, y, model_name: str, is_training: bool, class_names=None
):
    """
    Display comprehensive metrics for a fitted classifier.
    """
    # Predictions
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)

    classes = model.classes_

    if class_names is None:
        class_names = [str(c) for c in classes]

    print("=" * 65)
    print(
        f"CLASSIFICATION METRICS - {model_name} - {'TRAINING' if is_training else 'TESTING'}"
    )
    print("=" * 65)

    accuracy = accuracy_score(y, y_pred)
    precision_macro = precision_score(y, y_pred, average="macro", zero_division=0)
    precision_weighted = precision_score(y, y_pred, average="weighted", zero_division=0)
    recall_macro = recall_score(y, y_pred, average="macro", zero_division=0)
    recall_weighted = recall_score(y, y_pred, average="weighted", zero_division=0)
    f1_macro = f1_score(y, y_pred, average="macro", zero_division=0)
    f1_weighted = f1_score(y, y_pred, average="weighted", zero_division=0)

    auc_score = roc_auc_score(y, y_proba[:, 1])
    print(f"\n{'Accuracy':<25} {accuracy:.4f}")
    print(f"{'AUC':<25} {auc_score:.4f}")

    print(f"{'Precision (macro)':<25} {precision_macro:.4f}")
    print(f"{'Precision (weighted)':<25} {precision_weighted:.4f}")
    print(f"{'Recall (macro)':<25} {recall_macro:.4f}")
    print(f"{'Recall (weighted)':<25} {recall_weighted:.4f}")
    print(f"{'F1 (macro)':<25} {f1_macro:.4f}")
    print(f"{'F1 (weighted)':<25} {f1_weighted:.4f}")

    print(classification_report(y, y_pred, target_names=class_names, zero_division=0))
    print(confusion_matrix(y, y_pred))

    if not is_training:
        plot_roc_curves(y, y_proba)


def plot_roc_curves(y_test, y_proba):
    """Plot ROC curve(s) for binary or multiclass."""
    plt.figure(figsize=(8, 6))

    fpr, tpr, _ = roc_curve(y_test, y_proba[:, 1])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f"ROC (AUC = {roc_auc:.3f})")

    plt.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves")
    plt.legend(loc="lower right", fontsize=9)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
param_grid = {
    "logreg": {
        "clf__C": [0.01, 0.1, 1.0, 10.0, 50.0],
        "clf__l1_ratio": [0, 1],
        "clf__class_weight": ["balanced", {0: 1, 1: 1}, {0: 1, 1: 10}, {0: 1, 1: 100}],
    },
    "rf": {
        "clf__n_estimators": [100, 200],
        "clf__max_depth": [None, 10, 20, 40],
        "clf__class_weight": ["balanced", "balanced_subsample"],
        "clf__min_samples_leaf": [1, 2, 4],
        "clf__max_features": ["sqrt", "log2"],
    },
}

store_in: dict[tuple[str, str, str], GridSearchCV] = {}


def train_model(
    representation: str,
    model: str,
    scoring: str,
) -> GridSearchCV:
    if model == "logreg":
        clf = LogisticRegression(
            max_iter=5_000,
            solver="liblinear",
            random_state=RANDOM_STATE,
        )
    elif model == "rf":
        clf = RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE)

    if representation == "tfidf":
        feat = TfidfVectorizer(sublinear_tf=True)
        pipeline = Pipeline([("feat", feat), ("clf", clf)])
    elif representation == "sbert":
        pipeline = Pipeline([("clf", clf)])

    gs = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid[model],
        scoring=scoring,
        cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE),
        n_jobs=optimal_jobs,
        return_train_score=True,
        error_score="raise",
    )

    X_train = X_train_lemma if representation == "tfidf" else X_train_encoded
    X_test = X_test_lemma if representation == "tfidf" else X_test_encoded

    gs.fit(X_train, y_train)
    store_in[(model, representation, scoring)] = gs
    model_name = f"{representation.upper()} + {model} ({scoring}) "

    print(f"{model_name} - Best params: {gs.best_params_}\n")
    evaluate_classifier(gs.best_estimator_, X_train, y_train, model_name, True)
    print()
    evaluate_classifier(gs.best_estimator_, X_test, y_test, model_name, False)

### **5.1 Entrenamiento de Modelos con TF-IDF**

#### **5.1.1 Entrenamiento de Regresión Logística**

In [ ]:
for scoring in ['accuracy', 'recall', 'f1', 'precision']:
    train_model(representation="tfidf", model="logreg", scoring=scoring)

#### **5.1.2 Entrenamiento de Randon Forest**

In [ ]:
for scoring in ['accuracy', 'recall', 'f1', 'precision']:
    train_model(representation="tfidf", model="rf", scoring=scoring)

### **5.2 Entrenamiento de Modelos con Transfer Learning**

#### **5.2.1 Entrenamiento de Regresión Logística**

In [ ]:
for scoring in ['accuracy', 'recall', 'f1', 'precision']:
    train_model(representation="sbert", model="logreg", scoring=scoring)

#### **5.2.2 Entrenamiento de Random Forest**

In [ ]:
for scoring in ['accuracy', 'recall', 'f1', 'precision']:
    train_model(representation="sbert", model="rf", scoring=scoring)

## **6. Test de Hipótesis**

In [ ]:
from sklearn.model_selection import cross_val_score
from scipy.stats import wilcoxon

alpha = 0.05
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)

cross_val_score_acc: dict[tuple[str, str, str], Any] = {}


def calculate_cross_val_score_acc(model: tuple[str, str, str]):
    return cross_val_score(
        estimator=store_in[model].best_estimator_,
        X=X_train_lemma if model[1] == "tfidf" else X_train_encoded,
        y=y_train,
        scoring="accuracy",
        cv=cv,
        n_jobs=-1,
    )


def compare_models(
    first_model: tuple[str, str, str], second_model: tuple[str, str, str]
):
    acc_first_model = cross_val_score_acc.get(
        first_model, calculate_cross_val_score_acc(first_model)
    )
    cross_val_score_acc[first_model] = acc_first_model

    acc_second_model = cross_val_score_acc.get(
        second_model, calculate_cross_val_score_acc(second_model)
    )
    cross_val_score_acc[second_model] = acc_second_model

    if np.array_equal(acc_first_model, acc_second_model):
        print("Models are identical; p-value is effectively 1.0")
        return

    stat_m, p_value_m = wilcoxon(acc_first_model, acc_second_model)
    print("\n=== Wilcoxon signed-rank test ===")
    print(f"  Statistic: {stat_m:.4f}")
    print(f"  p-value  : {p_value_m:.6f}\n")

    if p_value_m < alpha:
        first = f"{first_model[1].upper()}-{first_model[0]}"
        second = f"{second_model[1].upper()}-{second_model[0]}"

        winner_m = first if acc_first_model.mean() > acc_second_model.mean() else second
        losser_m = second if winner_m == first else first
        print(
            f"Conclusión: rechazamos H₀ (p < {alpha}). {winner_m} supera estadísticamente al {losser_m}"
        )
    else:
        print(
            f"Conclusión: no rechazamos H₀ (p ≥ {alpha}). Los dos modelos son estadísticamente equivalentes en accuracy."
        )

La evaluación de los modelos vendrá dada de la siguiente forma:
1. Por cada grupo, es decir, por modelo y por feature extraction; es decir, el "mejor" modelo de regresión logística con TF-IDF . Se escoge a aquel modelo que presente un accurarcy >= 0.95 y que detecte de forma considerada la clase minoritaria. 

2. Por test de wilcoxon, comparar a los modelos del mismo grupo de feature extraction. Por ejemplo, del grupo de TF-IDF, comparar entre Regresión Logística y Random Forest.

3. Finalmente, por test de wilcoxon, comparar a los dos modelos finalistas.

In [ ]:
compare_models(("logreg", "tfidf", "accuracy"), ("rf", "tfidf", "f1"))

In [ ]:
compare_models(('logreg', 'sbert', 'f1'), ('rf', 'sbert', 'precision'))

In [ ]:
compare_models(("logreg", "tfidf", "accuracy"), ('rf', 'sbert', 'precision'))

Como podemos observar, entre los modelos finalistas, al consdierar el $p-value$, no existe diferencia alguna.

## **7. Curva de Aprendizaje y Ajuste de Lambda**

In [ ]:
from sklearn.model_selection import learning_curve


def plot_learning_curve(estimator, X, y, name):
    train_sizes, train_scores, test_scores = learning_curve(
        estimator,
        X,
        y,
        cv=cv,
        n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring="accuracy",
    )

    # Calculate mean and standard deviation
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    test_mean = np.mean(test_scores, axis=1)
    test_std = np.std(test_scores, axis=1)

    # Plotting
    plt.figure(figsize=(7, 5))
    plt.plot(train_sizes, train_mean, "o-", color="r", label="Training score")
    plt.plot(train_sizes, test_mean, "o-", color="g", label="Cross-validation score")

    # Optional: Fill area to show variance
    plt.fill_between(
        train_sizes,
        train_mean - train_std,
        train_mean + train_std,
        alpha=0.1,
        color="r",
    )
    plt.fill_between(
        train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color="g"
    )

    plt.title(f"Learning Curve for {name}")
    plt.xlabel("Training Examples")
    plt.ylabel("Score (Accuracy)")
    plt.legend(loc="best")
    plt.grid()
    plt.show()

In [ ]:
plot_learning_curve(store_in[("logreg", "tfidf", "accuracy")].best_estimator_, X_train_lemma, y_train, "TFIDF + Logistic Regression")

In [ ]:
plot_learning_curve(store_in[('rf', 'sbert', 'precision')].best_estimator_, X_train_encoded, y_train, "SBERT + Random Forest")

Como podemos observar, el rendimiento es perfecto y constante para ambos modelos con la data de entrenamiento. A primera instancia, puede parecer que sufren de alta varianza. Sin embargo, puesto que esto es un problema con clases imbalanceadas, podemos señalar que el modelo aprende a identificar clase mayoritaria y minoritaria en los datos de entrenamiento. No obstante, no es capaz de reconocer la clase minoritaria con los datos de validación. 

In [ ]:
grid_search = store_in[("logreg", "tfidf", "accuracy")]

results_df = pd.DataFrame(grid_search.cv_results_)
results_df.columns = [c.replace('param_clf__', '') if c.startswith('param_clf__') else c for c in results_df.columns]

best_index = grid_search.best_index_
best_l1 = results_df.loc[best_index, 'l1_ratio']
best_weight = results_df.loc[best_index, 'class_weight']

print(f"Filtering by best fixed params: l1_ratio={best_l1}, class_weight={best_weight}")

mask = (results_df['l1_ratio'] == best_l1) & (results_df['class_weight'] == best_weight)
filt_df = results_df[mask].sort_values('C')

lambdas = 1 / filt_df['C'].astype(float)
train_loss = 1 - filt_df['mean_train_score']
cv_loss = 1 - filt_df['mean_test_score']

plt.figure(figsize=(7, 5))
plt.plot(lambdas, train_loss, 'bo-', label=f'Train Loss (l1_ratio={best_l1})')
plt.plot(lambdas, cv_loss, 'mo-', label=f'CV Loss (l1_ratio={best_l1})')

best_lambda = 1 / grid_search.best_params_['clf__C']
plt.axvline(best_lambda, color='g', linestyle='--', label=f'Best lambda \approx {best_lambda:.2f}$')

plt.xscale('log')
plt.xlabel('Regularization Strength ($\lambda = 1/C$)')
plt.ylabel('Loss (1 - Accuracy)')
plt.title(f'Lambda Optimization (Fixed: Weight={best_weight})')
plt.legend()
plt.grid(True)
plt.show()

## **8. Conclusiones**